In [0]:
# =========================================================
# 00_PIPELINE_START
# Creates a unique run ID and initializes pipeline audit
# =========================================================

from datetime import datetime, timezone
import uuid


AUDIT_TABLE = (
    "training_lab_classic."
    "cold_chain.pipeline_run_audit"
)


# ---------------------------------------------------------
# GENERATE DYNAMIC RUN ID
# ---------------------------------------------------------

RUN_ID = (
    "CC_"
    + datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S"
    )
    + "_"
    + uuid.uuid4().hex[:6].upper()
)


print("Generated Pipeline Run ID:", RUN_ID)


# ---------------------------------------------------------
# CREATE INITIAL AUDIT RECORD
# ---------------------------------------------------------

spark.sql(f"""
INSERT INTO {AUDIT_TABLE}
(
    run_id,
    orchestrator,
    status,

    bronze_status,
    silver_status,
    gold_status,

    failed_stage,
    failed_task,
    error_type,
    error_message,
    retry_count,

    started_at,
    completed_at,

    input_rows,
    accepted_rows,
    rejected_rows,
    rejected_percent,

    temperature_exception_count,
    gold_temperature_window_rows,
    shipment_health_rows,
    delivery_performance_rows
)
VALUES
(
    '{RUN_ID}',
    'databricks',
    'RUNNING',

    'PENDING',
    'NOT_RUN',
    'NOT_RUN',

    NULL,
    NULL,
    NULL,
    NULL,
    0,

    current_timestamp(),
    NULL,

    0,
    0,
    0,
    NULL,

    0,
    0,
    0,
    0
)
""")


# ---------------------------------------------------------
# PASS RUN ID TO DOWNSTREAM TASKS
# ---------------------------------------------------------

dbutils.jobs.taskValues.set(
    key="run_id",
    value=RUN_ID
)


print("==============================================")
print("PIPELINE AUDIT INITIALIZED")
print("==============================================")
print("Run ID :", RUN_ID)
print("Status : RUNNING")